# 01 · Visão geral dos dados

Este caderno apresenta **que dados existem** no estudo, sem expor nenhum dado bruto, identificador de motorista ou coordenada individual.

* **Linha sintética** (pública, reproduzível sem credenciais): rede viária OpenStreetMap + demanda gerada por semente.
* **Linha histórica** (privada, V1 Viagens): só números **agregados** já publicados em `reports/atribuicao_final/`. Os dados brutos ficam em `data/extraido/` (ignorado pelo Git) e exigem `TCC_REAL_DATA_AUTHORIZED=YES`, liberado só pelo proprietário.

Limitações: a demanda histórica é o que foi registrado como solicitação (não é toda a procura); o km é **estimado** (reta × 1,3012), não odômetro.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "execute o notebook dentro do repositório"
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
SEED = 20261108
np.random.seed(SEED)

## Catálogo dos 22 cenários sintéticos
Os cenários compartilham estrutura (mesma rede, 100 veículos, mesma geração de pedidos); variam fatores controlados. **Não são 22 confirmações independentes.**

In [2]:
cat = json.loads((ROOT / "configs/scenarios/catalog.json").read_text(encoding="utf-8"))
print(cat["nota"])
df = pd.DataFrame(
    [
        {
            "id": c["id"],
            "grupo": c["grupo"],
            "finalidade": c["finalidade"],
            "pedidos_por_veiculo": c["parametros"]["pedidos_por_veiculo"],
            "espacial": c["parametros"]["espacial"],
            "res_zona": c["parametros"]["res_zona"],
        }
        for c in cat["cenarios"]
    ]
)
assert len(df) == 22 and df.id.is_unique
assert (df.finalidade == "confirmatorio_principal").sum() == 5
df

Os 22 cenários compartilham a mesma estrutura (rede, frota de 100 veículos, geração de pedidos) e variam fatores controlados; não são 22 confirmações independentes. As sementes 20001–20040 foram examinadas antes da decisão de adotar o método.


                      id                                     grupo                 finalidade  pedidos_por_veiculo  espacial  res_zona
0                     S1                                 principal    confirmatorio_principal                  8.0  uniforme         7
1                     S2                                 principal    confirmatorio_principal                  8.0      hubs         7
2                     S3                                 principal    confirmatorio_principal                  8.0      hubs         7
3                     S4                                 principal    confirmatorio_principal                  8.0      hubs         7
4                     S5                                 principal    confirmatorio_principal                  8.0     movel         7
5     S2_frota_abundante                       sensibilidade_frota  exploratorio_complementar                  4.0      hubs         7
6       S2_frota_escassa                       sensibil

## Volumes por período (agregados da linha histórica)
Fonte: `reports/atribuicao_final/volumes_por_periodo.json` (somente números agregados).

In [3]:
vol_path = ROOT / "reports/atribuicao_final/volumes_por_periodo.json"
if vol_path.exists():
    vol = pd.DataFrame(json.loads(vol_path.read_text(encoding="utf-8"))).T
    display(vol[["dias", "pedidos", "mot_min", "mot_med", "mot_max", "compromissos"]])
    assert int(vol.loc["final", "pedidos"]) == 23523 and int(vol.loc["replicacao", "pedidos"]) == 16918
else:
    print("Volumes agregados ausentes (resultados congelados não estão neste clone).")

            dias  pedidos  mot_min     mot_med  mot_max  compromissos
treino      43.0  51909.0     58.0  106.790698    132.0        1458.0
validacao   16.0  20178.0     60.0  111.312500    130.0         675.0
replicacao  15.0  16918.0     49.0   86.666667    100.0         674.0
final       25.0  23523.0     39.0   74.320000     95.0         842.0

## Dados brutos privados: disponibilidade neste ambiente

In [4]:
import os

raw = ROOT / "data/extraido/v1_viagens"
print("dados locais presentes:", raw.exists())
print("portão TCC_REAL_DATA_AUTHORIZED:", os.getenv("TCC_REAL_DATA_AUTHORIZED") == "YES")
if not raw.exists():
    print(
        "Sem os dados privados, somente as seções sintéticas e os agregados congelados podem ser reproduzidos; "
        "peça ao proprietário a extração local (scripts/extrair_v1_*.sh) para reexecutar a linha histórica."
    )

dados locais presentes: True
portão TCC_REAL_DATA_AUTHORIZED: False


## Rede e grade H3 do cenário sintético

In [5]:
from tcc_frota.scenarios import runner

world = runner.build_world(7)
print("pontos de parada:", len(world.pontos), "| zonas H3 (res. 7):", len(world.zonas))
assert len(world.pontos) == 368 and len(world.zonas) == 75

pontos de parada: 368 | zonas H3 (res. 7): 75
